# 09 Recommendations


This notebook generates evidence-based recommendations for improving the candidate's CV and professional profile in relation to a selected IT job posting.

The notebook uses outputs from previous steps:

- structured CV extraction
- structured job posting extraction
- CV quality analysis
- hybrid CV-job matching result

The goal is not to calculate a new score, but to explain what the candidate should improve in order to better match the selected job posting.

The recommendations are generated using an LLM with structured output.


## 1. Imports and Settings

In [32]:
import os
import json

from pathlib import Path
from datetime import datetime
from typing import List, Optional

from dotenv import load_dotenv
from pydantic import BaseModel, Field

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

In [33]:
CV_EXTRACTION_DIR = Path("../outputs/cv_extraction")
JOB_EXTRACTION_DIR = Path("../outputs/job_extraction")
CV_QUALITY_DIR = Path("../outputs/cv_quality")
MATCHING_DIR = Path("../outputs/matching")
RECOMMENDATIONS_DIR = Path("../outputs/recommendations")

In [34]:
structured_cv_path = CV_EXTRACTION_DIR / "structured_cv.json"
structured_job_path = JOB_EXTRACTION_DIR / "structured_job.json"
cv_quality_path = CV_QUALITY_DIR / "cv_quality_analysis.json"
matching_result_path = MATCHING_DIR / "matching_result.json"

In [35]:
recommendations_path = RECOMMENDATIONS_DIR / "recommendations.json"
recommendations_report_path = RECOMMENDATIONS_DIR / "recommendations_report.md"

In [36]:
load_dotenv(dotenv_path=Path("../.env"))

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

## 2. LLM Settings

In [37]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

## 3. Load Input Data

In [38]:
with open(structured_cv_path, "r", encoding="utf-8") as file:
    structured_cv = json.load(file)

In [39]:
with open(structured_job_path, "r", encoding="utf-8") as file:
    structured_job = json.load(file)

In [40]:
with open(cv_quality_path, "r", encoding="utf-8") as file:
    cv_quality_analysis = json.load(file)

In [41]:
with open(matching_result_path, "r", encoding="utf-8") as file:
    matching_result = json.load(file)

## 4. Define Recommendation Output Schema

In [42]:
class RecommendationItem(BaseModel):
    title: str = Field(
        description=(
            "Short title of the recommendation. "
            "The title should clearly describe what should be improved or developed."
        )
    )

    recommendation: str = Field(
        description=(
            "Concrete recommendation written in practical language. "
            "The recommendation should explain what the candidate should do. "
            "Do not suggest adding false information to the CV."
        )
    )

    reason: str = Field(
        description=(
            "Explanation of why this recommendation is important for the selected job posting or CV quality. "
            "The reason must be based on the provided CV, job posting, matching result or CV quality analysis."
        )
    )

    evidence: str = Field(
        description=(
            "Evidence from the input data that supports this recommendation. "
            "This may refer to a missing skill, weak CV quality area, missing responsibility evidence, or job requirement. "
            "If the evidence is that something is not present in the CV, clearly state that it is not evidenced."
        )
    )

    priority: str = Field(
        description=(
            "Priority level of the recommendation. "
            "Use one of the following values: High, Medium, Low. "
            "High priority should be used for missing required skills, major CV weaknesses or important job requirements."
        )
    )


In [43]:
class SkillDevelopmentRecommendation(BaseModel):
    skill: str = Field(
        description=(
            "Name of the missing or weakly evidenced skill, tool, technology or knowledge area."
        )
    )

    current_status: str = Field(
        description=(
            "Current status based on the CV and matching result. "
            "Use phrases such as 'not evidenced in CV', 'partially evidenced', or 'unclear from CV'. "
            "Do not claim that the candidate has the skill unless it is evidenced."
        )
    )

    development_recommendation: str = Field(
        description=(
            "Concrete recommendation for developing or improving this skill. "
            "The recommendation can include learning, practice, small projects, documentation or portfolio work."
        )
    )

    cv_update_recommendation: str = Field(
        description=(
            "Recommendation for how this skill should be represented in the CV. "
            "If the candidate already has this skill but it is not clearly shown, use wording such as: "
            "'If this is true, add it clearly to the CV with project or experience evidence.' "
            "Never recommend falsely adding a skill."
        )
    )

    priority: str = Field(
        description="Priority level: High, Medium, or Low."
    )


In [44]:
class PriorityAction(BaseModel):
    action: str = Field(
        description=(
            "A concrete next action the candidate should take."
        )
    )

    expected_impact: str = Field(
        description=(
            "Expected impact of this action on CV quality or job matching."
        )
    )

    priority: str = Field(
        description="Priority level: High, Medium, or Low."
    )


In [45]:
class RecommendationOutput(BaseModel):
    overall_recommendation_summary: str = Field(
        description=(
            "Concise summary of the most important recommendations for improving the CV and candidate profile "
            "in relation to the selected job posting."
        )
    )

    cv_improvement_recommendations: List[RecommendationItem] = Field(
        default_factory=list,
        description=(
            "Recommendations focused on improving the CV document itself. "
            "Examples include improving structure, adding clearer project descriptions, adding measurable results, "
            "clarifying experience, or better grouping technical skills."
        )
    )

    missing_required_skills_recommendations: List[SkillDevelopmentRecommendation] = Field(
        default_factory=list,
        description=(
            "Recommendations for required job skills that are missing or not clearly evidenced in the CV. "
            "This should be based primarily on rule-based missing required skills from the matching result."
        )
    )

    technical_development_recommendations: List[SkillDevelopmentRecommendation] = Field(
        default_factory=list,
        description=(
            "Recommendations for technical skills, tools, frameworks, databases, cloud tools or other technologies "
            "that the candidate should develop to improve job fit."
        )
    )

    project_recommendations: List[RecommendationItem] = Field(
        default_factory=list,
        description=(
            "Recommendations for project work that could help the candidate demonstrate missing or weakly evidenced skills. "
            "Do not invent projects that the candidate already completed. Suggest possible future projects or portfolio improvements."
        )
    )

    soft_skills_recommendations: List[RecommendationItem] = Field(
        default_factory=list,
        description=(
            "Recommendations for better evidencing soft skills requested in the job posting. "
            "Do not suggest writing generic claims such as 'excellent communication skills'. "
            "Suggest showing soft skills through experience, responsibilities, teamwork, mentoring, documentation or project examples."
        )
    )

    priority_actions: List[PriorityAction] = Field(
        default_factory=list,
        description=(
            "Short prioritized list of the most important next actions. "
            "These actions should be practical and based on the strongest gaps found in the analysis."
        )
    )

    evidence_based_notes: List[str] = Field(
        default_factory=list,
        description=(
            "Important notes about the limitations of the recommendation. "
            "Use this field to mention that some skills or experience are not evidenced in the CV, "
            "or that some recommendations depend on whether the candidate actually has the experience."
        )
    )

## 5. Define Recommendation Prompt

In [46]:
recommendation_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
You are an AI assistant specialized in CV improvement and career development recommendations for IT job matching.

Your task is to generate evidence-based recommendations using:
- structured CV data,
- structured job posting data,
- CV quality analysis,
- CV-job matching result.

Important rules:
- Use only the information provided in the input data.
- Do not invent candidate skills, experience, projects, education, certificates or achievements.
- Do not suggest adding false information to the CV.
- If a skill or experience is not evidenced in the CV, clearly say that it is not evidenced.
- If the candidate may have a skill but it is not visible in the CV, use wording such as: "If this is true, add it clearly to the CV with evidence."
- For missing required skills, recommend learning, practicing, building a project, or adding evidence only if true.
- Recommendations should be concrete, practical and useful.
- Prioritize required job skills and important CV quality weaknesses.
- Do not calculate a new match score.
- Do not override the previous matching result.
- Return the result using the required structured schema.

Recommendation style:
- Be clear and professional.
- Avoid generic advice.
- Explain why each recommendation matters.
- Connect recommendations to job requirements and CV evidence.
"""
        ),
        (
            "human",
            """
Generate recommendations using the following data.

Structured CV:

{structured_cv}

Structured job posting:

{structured_job}

CV quality analysis:

{cv_quality_analysis}

Matching result:

{matching_result}
"""
        )
    ]
)

## 6. Run LLM Recommendation Generation

In [47]:
structured_recommendation_llm = llm.with_structured_output(RecommendationOutput)

In [48]:
recommendation_chain = recommendation_prompt | structured_recommendation_llm

In [49]:
recommendation_result = recommendation_chain.invoke(
    {
        "structured_cv": json.dumps(structured_cv, indent=2, ensure_ascii=False),
        "structured_job": json.dumps(structured_job, indent=2, ensure_ascii=False),
        "cv_quality_analysis": json.dumps(cv_quality_analysis, indent=2, ensure_ascii=False),
        "matching_result": json.dumps(matching_result, indent=2, ensure_ascii=False)
    }
)

In [50]:
recommendation_dict = recommendation_result.model_dump()

## 7. Add Metadata to Recommendation Result

In [51]:
final_result = matching_result.get("final_result", {})
score_breakdown = matching_result.get("score_breakdown", {})
missing_items = matching_result.get("missing_items", {})
semantic_analysis = matching_result.get("semantic_analysis", {})

In [52]:
recommendation_output = {
    "metadata": {
        "created_at": datetime.now().isoformat(timespec="seconds"),
        "recommendation_type": "llm_based_evidence_recommendations",
        "llm_model": "gpt-4o-mini",
        "cv_source": str(structured_cv_path),
        "job_source": str(structured_job_path),
        "cv_quality_source": str(cv_quality_path),
        "matching_source": str(matching_result_path)
    },
    "job_information": {
        "job_title": structured_job.get("job_title"),
        "company_name": structured_job.get("company_name"),
        "job_category": structured_job.get("job_category"),
        "location": structured_job.get("location"),
        "work_mode": structured_job.get("work_mode"),
        "employment_type": structured_job.get("employment_type")
    },
    "matching_summary": {
        "final_hybrid_score": final_result.get("final_hybrid_score"),
        "rule_based_score": final_result.get("rule_based_score"),
        "semantic_score": final_result.get("semantic_score"),
        "match_category": final_result.get("match_category")
    },
    "recommendations": recommendation_dict
}

## 8. Save Recommendations as JSON

In [53]:
with open(recommendations_path, "w", encoding="utf-8") as file:
    json.dump(recommendation_output, file, indent=4, ensure_ascii=False)

## 9. Define Markdown Helper Functions

In [54]:
def create_recommendation_section(items, empty_message="- No recommendations generated."):
    if not items:
        return empty_message

    lines = []

    for index, item in enumerate(items, start=1):
        title = item.get("title", f"Recommendation {index}")
        recommendation = item.get("recommendation", "")
        reason = item.get("reason", "")
        evidence = item.get("evidence", "")
        priority = item.get("priority", "")

        lines.append(f"### {index}. {title}")
        lines.append("")
        lines.append(f"- Priority: {priority}")
        lines.append(f"- Recommendation: {recommendation}")
        lines.append(f"- Reason: {reason}")
        lines.append(f"- Evidence: {evidence}")
        lines.append("")

    return "\n".join(lines)

In [55]:
def create_skill_recommendation_section(items, empty_message="- No skill development recommendations generated."):
    if not items:
        return empty_message

    lines = []

    for index, item in enumerate(items, start=1):
        skill = item.get("skill", f"Skill {index}")
        current_status = item.get("current_status", "")
        development_recommendation = item.get("development_recommendation", "")
        cv_update_recommendation = item.get("cv_update_recommendation", "")
        priority = item.get("priority", "")

        lines.append(f"### {index}. {skill}")
        lines.append("")
        lines.append(f"- Priority: {priority}")
        lines.append(f"- Current status: {current_status}")
        lines.append(f"- Development recommendation: {development_recommendation}")
        lines.append(f"- CV update recommendation: {cv_update_recommendation}")
        lines.append("")

    return "\n".join(lines)

In [56]:
def create_priority_actions_section(items, empty_message="- No priority actions generated."):
    if not items:
        return empty_message

    lines = []

    for index, item in enumerate(items, start=1):
        action = item.get("action", "")
        expected_impact = item.get("expected_impact", "")
        priority = item.get("priority", "")

        lines.append(f"{index}. {action}")
        lines.append(f"   - Priority: {priority}")
        lines.append(f"   - Expected impact: {expected_impact}")

    return "\n".join(lines)

In [57]:
def create_markdown_list(items, empty_message="- No items found."):
    if not items:
        return empty_message

    return "\n".join([f"- {item}" for item in items])

## 10. Generate Markdown Recommendations Report

In [58]:
recommendations = recommendation_output["recommendations"]

report = f"""
# CV Improvement and Professional Development Recommendations

## Job Information

- Job title: {structured_job.get("job_title")}
- Company: {structured_job.get("company_name")}
- Job category: {structured_job.get("job_category")}
- Location: {structured_job.get("location")}
- Work mode: {structured_job.get("work_mode")}
- Employment type: {structured_job.get("employment_type")}

## Matching Summary

- Final hybrid score: {final_result.get("final_hybrid_score")}/100
- Rule-based score: {final_result.get("rule_based_score")}/100
- LLM semantic score: {final_result.get("semantic_score")}/100
- Match category: {final_result.get("match_category")}

## Overall Recommendation Summary

{recommendations.get("overall_recommendation_summary")}

## CV Improvement Recommendations

{create_recommendation_section(recommendations.get("cv_improvement_recommendations", []))}

## Missing Required Skills Recommendations

{create_skill_recommendation_section(recommendations.get("missing_required_skills_recommendations", []))}

## Technical Development Recommendations

{create_skill_recommendation_section(recommendations.get("technical_development_recommendations", []))}

## Project Recommendations

{create_recommendation_section(recommendations.get("project_recommendations", []))}

## Soft Skills Recommendations

{create_recommendation_section(recommendations.get("soft_skills_recommendations", []))}

## Priority Actions

{create_priority_actions_section(recommendations.get("priority_actions", []))}

## Evidence-Based Notes

{create_markdown_list(recommendations.get("evidence_based_notes", []), empty_message="- No additional evidence-based notes.")}
"""

## 11. Save Markdown Recommendations Report

In [59]:
with open(recommendations_report_path, "w", encoding="utf-8") as file:
    file.write(report)